# Step 2: Main notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from method_b_catchment import extract_catchment_feature
from method_c_zonal import compute_zonal_stat
#from method_d_join import spatial_join_attribute

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

## Imports

#### Import des segments

In [2]:
# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments, replace file path -->")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-1'
segments_gdf = gpd.read_file(f"{file_path}/step1_pedestrian_segments.gpkg")

Loading pedestrian segments, replace file path -->


#### Import du csv - méthode de traitement des attributs

In [3]:
# #Attributes info
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input'
# If the file attributes_info exists, load it
if os.path.exists(f"{file_path}/attributs_info.csv"):
    attributs_info = pd.read_csv(f"{file_path}/attributs_info.csv")
else:
    # If it does not exist, create a new DataFrame with the required structure
    attributs_info = pd.DataFrame(columns=['attribute', 'method', 'how', 'value_column', 'buffer_size'])
    
    # Example data to fill the DataFrame
    # You can modify this part to include the actual attributes you want to process
    attributs_info = pd.DataFrame({
        'attribute': ['arbre', 'accident', 'vitesse'],
        'method': ['buffer', 'buffer', 'zonal'],
        'how' : ['count', 'sum', None],  # 'how' can be 'count', 'mean', etc.
        'value_column': [None, 'PIETONS', 'VITESSE'],  # Column to aggregate
        'buffer_size': [30, 50, 15]  # Example buffer size for method A
    })
    attributs_info.to_csv(f"{file_path}/attributs_info.csv", index=False)

#### Import des attributs et sauvegarde en gpkg (à modifier pour chaque attribut)

In [4]:
if not os.path.exists('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'):
    os.makedirs('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs')
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'


#Chargement de la couhe des accidents 
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/accident/OTC_ACCIDENTS-SHP'
accident_gdf  = gpd.read_file(f"{file_path}/OTC_ACCIDENTS.shp")
print('Couche accident chargée avec succès')
accident_gdf = accident_gdf.to_crs(2056)
accident_gdf.to_file(f"{save_path}/accident.gpkg", driver='GPKG')

#Chargement de la couche des arbre
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/arbre/SIPV_ICA_ARBRE_ISOLE-SHP'
arbre_gdf = gpd.read_file(f"{file_path}/SIPV_ICA_ARBRE_ISOLE.shp")
print('Couche arbre chargée avec succès')
arbre_gdf = arbre_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
arbre_gdf.to_file(f"{save_path}/arbre.gpkg", driver='GPKG')

#Chargement de la couche des vitesse
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/vitesse/OTC_LIMITATIONS_VITESSE-SHP'
vitesse_gdf = gpd.read_file(f"{file_path}/OTC_LIMITATIONS_VITESSE.shp")
print('Couche vitesse chargée avec succès')
vitesse_gdf = vitesse_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
vitesse_gdf.to_file(f"{save_path}/vitesse.gpkg", driver='GPKG')

# Ajouter les autres couches d'attributs ici

Couche accident chargée avec succès
Couche arbre chargée avec succès
Couche vitesse chargée avec succès


## Traitement des attributs

In [5]:
vitesse_gdf.head()

,TYPE_LIMIT,TYPE_ZONE,NOM_ZONE,VITESSE,ANNEE_MES,geometry
0,Prescription 80 km/h,0,None,80,0,"MULTIPOLYGON (((2503567.178 1135218.627, 25035..."
1,Prescription 50 km/h,0,None,50,0,"POLYGON ((2503420.791 1134155.703, 2503426.943..."
2,Zone 30 km/h,1,Crêts de Pregny,30,2025,"POLYGON ((2499564.295 1121103.575, 2499511.072..."
3,Zone de rencontre,3,Cointrin,20,2002,"POLYGON ((2497314.067 1120141.116, 2497322.233..."
4,Zone piétonne,5,Vieille-Ville / Cathédrale,0,0,"POLYGON ((2500515.347 1117467.481, 2500504.337..."


In [6]:
speed_stats = compute_zonal_stat(
    segments_gdf=segments_gdf,
    features_gdf=vitesse_gdf,
    buffer_radius=15,
    value_column="VITESSE",
    stat="max"
)

100%|██████████| 255226/255226 [11:33<00:00, 368.07it/s] 


In [8]:
speed_stats.head(50)

,segment_id,zonal_max
0,000000,60.0
1,000001,60.0
2,000002,60.0
3,000003,60.0
4,000004,60.0
5,000005,60.0
6,000006,60.0
7,000007,60.0
8,000008,30.0
9,000009,0.0


In [7]:
speed_stats.head(50)

,segment_id,zonal_max
0,000000,60.0
1,000001,60.0
2,000002,60.0
3,000003,60.0
4,000004,60.0
5,000005,60.0
6,000006,60.0
7,000007,60.0
8,000008,30.0
9,000009,0.0


In [ ]:
# # Charger la table des attributs et méthodes
# attributs_info = pd.read_csv('../../Data/input/attributs_info.csv')

# # Charger les segments
# segments_gdf = gpd.read_file('../../Data/output/step-1/step1_pedestrian_segments.gpkg')

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segments_gdf
segments_gdf = segments_gdf[['osmid', 'maxspeed', 'geometry', 'segment_id']]

# Boucle sur chaque attribut
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    method = row['method']
    how = row['how']
    value_column = row['value_column']
    buffer_size = row['buffer_size']

    # Charger la couche attribut depuis gpd_attributs
    attribute_gdf = gpd.read_file(f"../../Data/output/step-2/gpkg_attributs/{attribute_name}.gpkg")
    attribute_gdf = attribute_gdf.to_crs(segments_gdf.crs)

    # Appliquer la méthode
    if method == "buffer":
        attribute_df = extract_buffer_feature(
            segments_gdf,
            attribute_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            how=how,
            value_column=value_column
        )
        print(f"Buffer feature extracted for {attribute_name} with method {method}")
    elif method == "catchment":
        attribute_df = extract_catchment_feature(
            segments_gdf=segments_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            points_gdf=attribute_gdf
        )
        print(f"Catchment feature extracted for {attribute_name} with method {method}")
    elif method == "zonal":
        attribute_df = compute_zonal_stat(
            segments_gdf,
            attribute_gdf,
            value_column=value_column
        )
        print(f"Zonal statistics computed for {attribute_name} with method {method}")
    # Ajoute d'autres méthodes si besoin

    # Ajouter la colonne au GeoDataFrame principal
    segments_gdf[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name]

# Sauvegarder le GeoDataFrame mis à jour
segments_gdf.to_csv('../../Data/output/step-2/step2_features.csv', index=False)

In [ ]:
pd.read_csv('../../Data/output/step-2/step2_features.csv').head(5)  # Afficher les 5 premières lignes pour vérification